In [2]:
import sqlite3 
import pandas as pd

conn = sqlite3.connect("../data/paysim.db")

In [4]:
query = """
SELECT type, Count(*) AS total_txns
from transactions
Group BY type
"""
pd.read_sql(query, conn)

,type,total_txns
0,CASH_IN,1399284
1,CASH_OUT,2237500
2,DEBIT,41432
3,PAYMENT,2151495
4,TRANSFER,532909


## Q2: Largest fraud transactions
The 10 largest frauds are all exactly 10,000,000 (the apparent transaction cap), and they occur in TRANSFER + CASH_OUT pairs of the same amount in the same hour. This points to a transfer-then-cash-out pattern.

In [9]:
query = """

SELECT type, nameOrig,amount, step
from transactions
where isFraud = 1 
order by amount DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,type,nameOrig,amount,step
0,TRANSFER,C7162498,10000000.0,4
1,CASH_OUT,C351297720,10000000.0,4
2,TRANSFER,C416779475,10000000.0,19
3,CASH_OUT,C2050703310,10000000.0,19
4,TRANSFER,C1439740840,10000000.0,33
5,CASH_OUT,C29118015,10000000.0,33
6,TRANSFER,C53057884,10000000.0,72
7,CASH_OUT,C1438388258,10000000.0,72
8,TRANSFER,C1237313447,10000000.0,82
9,CASH_OUT,C1079335762,10000000.0,82


## Q3: Fraud vs normal transaction amounts
Fraudulent CASH_OUT transactions average about 1.46M, compared with about 174K for normal ones (roughly 8x larger).
Fraudulent TRANSFER transactions average about 1.48M, compared with about 906K for normal ones (roughly 1.6x larger).

In [10]:
query = """

select type,isFraud, AVG(amount) AS avg_amount
from transactions
where type IN ('TRANSFER', 'CASH_OUT')
Group by type, isFraud
order by type, isFraud
"""
pd.read_sql(query,conn)

,type,isFraud,avg_amount
0,CASH_OUT,0,1.739172e+05
1,CASH_OUT,1,1.455103e+06
2,TRANSFER,0,9.062290e+05
3,TRANSFER,1,1.480892e+06


Do fraudulent transactions empty the sender's account?
Almost all fraudulent CASH_OUTs (99.98%) leave the sender at zero balance, but so do 88.7% of normal ones. For TRANSFER the rate is the same for fraud and normal (about 96%). Ending at zero balance alone is therefore a weak rule, so Q5 refines it by requiring the account to have had money before.

In [17]:
query = """
SELECT type,
       isFraud,
       COUNT(*) AS total_txns,
       SUM(CASE WHEN newbalanceOrig = 0 THEN 1 ELSE 0 END) AS emptied_accounts,
       ROUND(100.0 * SUM(CASE WHEN newbalanceOrig = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_emptied
FROM transactions
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY type, isFraud
ORDER BY type, isFraud
"""
pd.read_sql(query, conn)

,type,isFraud,total_txns,emptied_accounts,pct_emptied
0,CASH_OUT,0,2233384,1981096,88.70
1,CASH_OUT,1,4116,4115,99.98
2,TRANSFER,0,528812,507507,95.97
3,TRANSFER,1,4097,3938,96.12


In [16]:
import os
os.makedirs("../dashboard_data", exist_ok=True)

# 1. fraud by type
q1 = """
SELECT type, COUNT(*) AS total_txns, SUM(isFraud) AS fraud_txns
FROM transactions
GROUP BY type
"""
pd.read_sql(q1, conn).to_csv("../dashboard_data/fraud_by_type.csv", index=False)

# 2. fraud by day and type (step is in hours, so step / 24 gives the day)
q2 = """
SELECT step / 24 AS day, type, COUNT(*) AS total_txns, SUM(isFraud) AS fraud_txns
FROM transactions
GROUP BY step / 24, type
"""
pd.read_sql(q2, conn).to_csv("../dashboard_data/fraud_by_day.csv", index=False)